# Classification with a feed-forward network: iris species

In this notebook we predict a **class** (which of 3 iris species a flower belongs to)
from 4 flower measurements. Compare with the regression notebook (`S43_regression.ipynb`):
the steps are the same, but three things change for classification:

- the target is **one-hot encoded**: one column per class;
- the model has **one output per class**; these are raw scores (logits), with no activation;
- training uses **cross-entropy loss**, and we also track **accuracy**.

Steps: load and explore the data → one-hot encode the target → split into train/test →
standardize → build the model → train → evaluate on the test set.

In [ ]:
# ── 0. Imports ─────────────────────────────────────────────────────────────
import matplotlib.pyplot as plt
import pandas as pd  # tables (DataFrame) for exploring the data
import seaborn as sns  # quick statistical plots
import torch
from sklearn.datasets import load_iris  # built-in example dataset
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from torch import nn

In [ ]:
# ── 1. Data ────────────────────────────────────────────────────────────────
# The iris dataset: 150 flowers, 50 of each of 3 species
#   0 = setosa, 1 = versicolor, 2 = virginica
# Each flower has 4 features: sepal length/width and petal length/width (cm).
iris = load_iris()

In [ ]:
print(iris.DESCR)  # full description of the dataset

In [ ]:
# Put the features and the target into one table so they are easy to explore.
# Each row is one flower; the last column ("target") is the species (0, 1 or 2).
df = pd.DataFrame(data=iris.data, columns=iris.feature_names)
df["target"] = iris.target
df

In [ ]:
# ── 2. Explore ─────────────────────────────────────────────────────────────
# Histogram of every column. The "target" histogram shows 3 equal bars:
# the classes are balanced (50 flowers each).
df.hist(bins=20, figsize=(12, 10))

In [ ]:
# Boxplot of the features (all columns except the target) on one axis.
# The features have different ranges; we standardize them in step 5.
df.iloc[:, :-1].boxplot()

In [ ]:
# ── 3. Inputs and one-hot targets ──────────────────────────────────────────
# Inputs: all columns except the last -> shape (150, 4)
# The leading "_" marks raw (not yet scaled) data.
_X = df.iloc[:, :-1].values

In [ ]:
# The target is a class number: 0, 1 or 2.
# We should NOT use it directly as a number to predict: that would tell the
# model that class 2 is "bigger" than class 1, and that class 1 sits
# "between" classes 0 and 2. Species have no such order.
df[["target"]]

In [ ]:
# One-hot encoding: one column per class, with 1 in the column of the
# true class and 0 elsewhere.
#   0 -> [1, 0, 0]
#   1 -> [0, 1, 0]
#   2 -> [0, 0, 1]
# The result has shape (150, 3): one target column per model output.
_Y = pd.get_dummies(df["target"]).astype(float).values

In [ ]:
# ── 4. Train/test split ────────────────────────────────────────────────────
# Keep 30% of the flowers aside as a test set. The model never sees them
# during training, so they tell us how well it works on new flowers.
# random_state=0 makes the split the same on every run.
_X_train, _X_test, _Y_train, _Y_test = train_test_split(
    _X, _Y, test_size=0.3, random_state=0
)

print(_X_train.shape)
print(_X_test.shape)
print(_Y_train.shape)
print(_Y_test.shape)

In [ ]:
# ── 5. Standardize ─────────────────────────────────────────────────────────
# StandardScaler turns each feature into: (value - mean) / std
# so every feature has mean 0 and standard deviation 1.
#
# fit_transform on the TRAIN set: learn mean/std from the training data.
# transform on the TEST set  : reuse the training mean/std.
# (Fitting on the test set would leak information about it into training.)
scX = StandardScaler()
X_train = scX.fit_transform(_X_train)
X_test = scX.transform(_X_test)

# Unlike regression, the targets are NOT scaled: one-hot vectors are
# already 0/1 and must stay that way. We only copy them, so the variable
# names match the regression notebook.
Y_train = _Y_train
Y_test = _Y_test

In [ ]:
# Compare with the earlier boxplot: all features are now centered at 0
# with a similar spread.
pd.DataFrame(data=X_train).boxplot()

In [ ]:
# PyTorch works with tensors, not NumPy arrays.
# .float() converts to float32: NumPy uses float64 by default, but the
# model's weights are float32, and the two types cannot be mixed.
X_train_pt = torch.from_numpy(X_train).float()
X_test_pt = torch.from_numpy(X_test).float()
Y_train_pt = torch.from_numpy(Y_train).float()
Y_test_pt = torch.from_numpy(Y_test).float()

In [ ]:
# ── 6. Model ───────────────────────────────────────────────────────────────
# Read the input and output sizes from the data instead of hard-coding them.
num_features = _X.shape[1]  # 4 inputs per flower
num_outputs = _Y.shape[1]  # 3 outputs: one score per class
print(num_features, num_outputs)

In [ ]:
# The same hidden layers as the regression model: 24 -> 12 -> 6 neurons,
# each followed by ReLU. Only the output layer differs: 3 outputs, one per class.
#
# Note: there is NO softmax at the end. The model outputs raw scores
# (logits), and CrossEntropyLoss applies the softmax internally.
model = nn.Sequential(
    nn.Linear(num_features, 24),  # hidden layer 1: inputs -> 24 neurons
    nn.ReLU(),
    nn.Linear(24, 12),  # hidden layer 2: 24 -> 12 neurons
    nn.ReLU(),
    nn.Linear(12, 6),  # hidden layer 3: 12 -> 6 neurons
    nn.ReLU(),
    nn.Linear(6, num_outputs),  # output layer: 6 neurons -> 1 logit per class
)

In [ ]:
# Count the trainable numbers (weights + biases) in the model.
# A Linear(a, b) layer has a*b weights and b biases:
#   4*24  + 24 = 120
#   24*12 + 12 = 300
#   12*6  + 6  =  78
#   6*3   + 3  =  21
#   total      = 519
pytorch_total_params = sum(p.numel() for p in model.parameters())
print(f"Total number of parameters in the model: {pytorch_total_params}")

In [ ]:
# ── 7. Train ───────────────────────────────────────────────────────────────
# One epoch = one pass over the whole training set. Here every epoch uses
# all training samples at once (full-batch gradient descent).
n_epochs = 500

# Adam is a variant of gradient descent that adapts the step size for
# each weight. Try changing the learning rate (lr) to see its effect:
#   too small -> slow training; too large -> loss may jump around.
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

# CrossEntropyLoss = Softmax + negative log-likelihood combined into one step.
#   - Input  : raw logits (any real numbers), NOT probabilities
#   - Target : here, one-hot vectors (a vector of class numbers also works)
#   - Idea   : softmax turns the 3 logits into probabilities that sum to 1;
#              the loss is -log(probability given to the true class).
#              It is 0 when that probability is 1 and grows as it drops.
loss_fn = nn.CrossEntropyLoss()

loss_arr = []  # training loss of each epoch, for plotting later
acc_arr = []  # training accuracy of each epoch
for epoch in range(n_epochs):

    model.train()  # training mode (matters for layers like Dropout; none here)
    Y_pred = model(X_train_pt)  # forward pass: logits, shape (105, 3)
    loss = loss_fn(Y_pred, Y_train_pt)  # compare logits with true classes

    optimizer.zero_grad()  # clear gradients from the previous step
    loss.backward()  # backpropagation: compute gradients
    optimizer.step()  # update weights

    epoch_train_loss = loss.item()  # .item() turns a 1-element tensor into a Python float
    loss_arr.append(epoch_train_loss)

    # Accuracy = fraction of samples whose predicted class is correct.
    # argmax(dim=1) picks the column with the largest value in each row:
    #   for the logits  -> the predicted class (no softmax needed: softmax
    #                      does not change which value is largest)
    #   for the one-hot -> the true class
    predicted_classes = torch.argmax(Y_pred, dim=1)
    true_classes = torch.argmax(Y_train_pt, dim=1)
    accuracy = (predicted_classes == true_classes).sum().item() / len(true_classes)
    acc_arr.append(accuracy)

    if epoch % 100 == 0:
        print(f"Epoch {epoch:3d}: loss = {epoch_train_loss:.4f}, accuracy = {accuracy:.4f}")

In [ ]:
# Training loss over the epochs: it should drop and then level off.
sns.lineplot(loss_arr)

In [ ]:
# Training accuracy over the epochs: it should rise towards 1.0 (100%).
# Accuracy moves in steps, because it only changes when a sample's
# predicted class flips.
sns.lineplot(acc_arr)

In [ ]:
# ── 8. Evaluate ────────────────────────────────────────────────────────────
# Compute the loss and accuracy on the test set, which the model has never seen.
# no_grad() turns off gradient tracking: we are only evaluating, not training.
with torch.no_grad():
    test_pred = model(X_test_pt)
    final_loss = loss_fn(test_pred, Y_test_pt)

    predicted_classes = torch.argmax(test_pred, dim=1)
    true_classes = torch.argmax(Y_test_pt, dim=1)
    final_accuracy = (predicted_classes == true_classes).sum().item() / len(true_classes)


In [ ]:
# Accuracy is easier to read than the loss: e.g. 0.978 means 44 of the
# 45 test flowers were classified correctly.
print(final_loss, final_accuracy)